In [ ]:
%pylab inline

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
import pandas as pd
from glob import glob
from tqdm import tqdm
from skimage import io
import pickle

In [ ]:
from skimage.transform import rotate, resize
from skimage.exposure import equalize_adapthist

In [ ]:
path_images = './00.data/cropped_old/'
path_labels = './RotatedImageFiles.txt'

In [ ]:
data_files = glob(path_images+'*')
data_files_zero = [path for path in data_files if '_0.jpg' in path]
data_labels = pd.read_csv(path_labels)

In [ ]:
data_labels.index = data_labels.Filename.values

In [ ]:
def get_image(img_path):
    
    img = io.imread(img_path)
    
    return(img)


def get_label(img_path,out=data_labels):
    
    img_id = img_path.split('/')[-1]
    row = out.loc[img_id]
    angle = row['Doppler_Angle']
    
    return(angle)

In [ ]:
data_x = [get_image(path) for path in data_files]
data_y = [get_label(path) for path in data_files]

In [ ]:
data_x_zero = [get_image(path) for path in data_files_zero]
data_y_zero = [get_label(path) for path in data_files_zero]

In [ ]:
def predition_generator(input_data,
                        output_data,
                        model_pretrained,
                        batch_size=25,
                        img_channels=3,
                        rotation_range=60,
                        rotation_step=5,
                        negative=False,
                        input_size=(100, 100),
                        equalize_hist=True):

    img_rows, img_cols = input_size

    while True:
        
        batch_files = input_data
        
        if negative:
            start_range = -1 * rotation_range
        else:
            start_range = 0
            
        angle_set = np.arange(start_range,rotation_range,rotation_step)
        
        batch_paths = []
        batch_images = []
        batch_labels = []
        batch_rotation = []

        for current_file in batch_files:
            for rotation_angle in angle_set:
                
                current_image = get_image(img_path = current_file)
                current_label = get_label(img_path=current_file, out=output_data)
                new_angle = current_label+rotation_angle
            
                if equalize_hist:
                    current_image = equalize_adapthist(image=current_image)
                if new_angle>180:
                    new_angle-=180

                current_image = resize(image=current_image, output_shape=(img_rows,img_cols), mode='reflect')
                current_image = rotate(image=current_image, angle=rotation_angle, mode='reflect')
                current_image = np.stack([current_image, current_image, current_image], axis=-1)
                
                batch_rotation.append(rotation_angle)
                batch_paths.append(current_file)
                batch_images.append(current_image)
                batch_labels.append(new_angle)
        
        batch_paths = np.array(batch_paths)
        batch_rotation = np.array(batch_rotation)
        batch_images = np.array(batch_images)
        batch_images = model_pretrained.predict(batch_images)
        batch_labels = np.array(batch_labels)
        
        #batch_images = np.expand_dims(batch_images,-1)
        batch_labels = np.expand_dims(np.expand_dims(np.expand_dims(batch_labels,1),1),1)
        
        batch_x, batch_y = batch_images, batch_labels

        return(batch_paths,batch_rotation,batch_x, batch_y)

In [ ]:
from keras import backend as K
from keras.models import Model
from keras.regularizers import l2
from keras.optimizers import Adam
from keras.layers import Input, BatchNormalization, Conv2D, Dense, Activation, Dropout

In [ ]:
from keras.applications import Xception,VGG16,VGG19,ResNet50
from keras.applications import InceptionV3, InceptionResNetV2
from keras.applications import MobileNet, DenseNet121, DenseNet169, DenseNet201, NASNetLarge

In [ ]:
variants = ['Xception', 'VGG16', 'VGG19', 'ResNet50', 
            'InceptionV3', 'InceptionResNetV2', 
            'DenseNet121', 'DenseNet169', 'DenseNet201']

In [ ]:
angle_range = 90
rotation_step = 1
dropout_rate = 0.25

In [ ]:
import pickle
import deepdish as dd

In [ ]:
for variant in variants:
    
    print(variant)
    run_version = 'model_'+variant+'_'+str(angle_range)
    features_file = './01.features/'+run_version+'_features.p'
    features_hdf = './01.features/'+run_version+'_features.h5'
    
    K.clear_session()
    
    print('Load Model :')
    
    if variant =='Xception':
        img_size = (139,139,3)
        model_pretrained = Xception(include_top=False,input_shape=img_size)

    elif variant =='VGG16':
        img_size = (139,139,3)
        model_pretrained = VGG16(include_top=False,input_shape=img_size)

    elif variant =='VGG19':
        img_size = (139,139,3)
        model_pretrained = VGG19(include_top=False,input_shape=img_size)

    elif variant =='ResNet50':
        img_size = (197,197,3)
        model_pretrained = ResNet50(include_top=False,input_shape=img_size)

    elif variant =='InceptionV3':
        img_size = (139,139,3)
        model_pretrained = InceptionV3(include_top=False,input_shape=img_size)

    elif variant =='InceptionResNetV2':
        img_size = (139,139,3)
        model_pretrained = InceptionResNetV2(include_top=False,input_shape=img_size)

    elif variant =='DenseNet121':
        img_size = (221,221,3)
        model_pretrained = DenseNet121(include_top=False,input_shape=img_size)

    elif variant =='DenseNet169':
        img_size = (221,221,3)
        model_pretrained = DenseNet169(include_top=False,input_shape=img_size)

    elif variant =='DenseNet201':
        img_size = (221,221,3)
        model_pretrained = DenseNet201(include_top=False,input_shape=img_size)

    elif variant =='NASNetLarge':
        img_size = (331,331,3)
        model_pretrained = NASNetLarge(include_top=False,input_shape=img_size)


    model_pretrained._make_predict_function()
    
    print('Creating Features :')
    test_features = predition_generator(input_data=data_files_zero,
                                         output_data=data_labels,
                                         model_pretrained=model_pretrained,
                                         input_size=img_size[:2],
                                         rotation_range=angle_range,
                                         rotation_step=rotation_step,
                                         negative=True)
    
    
    data_paths, data_rotation, data_x, data_y = test_features
    
    print('Saving Features : '+features_hdf)
    data_to_save = {
                    'data_paths':data_paths, 
                    'data_rotation':data_rotation, 
                    'data_x':data_x, 
                    'data_y':data_y
                    }
    
    dd.io.save(features_hdf, data_to_save)

In [ ]:
dd.io.save( './01.features/model_VGG16_90_features_compresssed.h5',data_to_save)

In [ ]:
data_test_loaded = dd.io.load('./01.features/model_VGG16_90_features_compresssed.h5')

In [ ]:
data_paths = data_test_loaded['data_paths']
data_rotation = data_test_loaded['data_rotation']
data_x = data_test_loaded['data_x']
data_y = data_test_loaded['data_y']